# Data quality check

Verifies that normalized funding parquets exist for each venue and that the
schema, time coverage, and gap distribution are reasonable. Run after
`python -m funding_arb.main_collect ...`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path('..').resolve()
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.utils.io import normalized_dir

files = list(normalized_dir().glob('funding_*.parquet'))
files

In [ ]:
def summarize(p):
    df = pd.read_parquet(p)
    return {
        'file': p.name,
        'rows': len(df),
        'venues': df['venue'].unique().tolist() if 'venue' in df else [],
        'symbols': df['symbol'].nunique() if 'symbol' in df else 0,
        'start': df['timestamp_utc'].min() if 'timestamp_utc' in df else None,
        'end':   df['timestamp_utc'].max() if 'timestamp_utc' in df else None,
    }

pd.DataFrame([summarize(f) for f in files])

In [ ]:
# Gap distribution per (venue, symbol)
for p in files:
    df = pd.read_parquet(p)
    if df.empty: continue
    for (v, s), sub in df.groupby(['venue','symbol']):
        ts = sub.sort_values('timestamp_utc')['timestamp_utc']
        gaps = ts.diff().dt.total_seconds().dropna() / 3600.0
        if gaps.empty: continue
        print(f'{v}/{s}: median_gap_h={gaps.median():.2f} max_gap_h={gaps.max():.2f}')